# 02 - Validacao PyTorch

Recriacao da MLP com `nn.Linear` e comparacao das curvas com a implementacao NumPy.

In [9]:
import sys
from pathlib import Path

for src_path in [Path.cwd() / 'src', Path.cwd().parent / 'src', Path('/content/ap2-ia/src')]:
    if src_path.exists():
        sys.path.insert(0, str(src_path))
        break

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from medmnist import PathMNIST
from utils import set_seed

set_seed(42)

In [10]:
class TorchMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 9))
    def forward(self, x):
        return self.net(x)

train_ds = PathMNIST(split='train', size=28, download=True)
x_np = train_ds.imgs.astype('float32') / 255.0
x_np = x_np.mean(axis=-1)  # RGB 28x28x3 -> grayscale 28x28, mesma entrada 784 da MLP NumPy
x = torch.tensor(x_np, dtype=torch.float32).flatten(1)
y = torch.tensor(train_ds.labels.reshape(-1), dtype=torch.long)
loader = DataLoader(TensorDataset(x, y), batch_size=128, shuffle=True)

model = TorchMLP()
opt = torch.optim.SGD(model.parameters(), lr=1e-2, momentum=0.9)
criterion = nn.CrossEntropyLoss()
history_torch = []

for epoch in range(20):
    losses = []
    for xb, yb in loader:
        opt.zero_grad(set_to_none=True)
        loss = criterion(model(xb), yb)
        loss.backward()
        opt.step()
        losses.append(loss.item())
    with torch.no_grad():
        pred = model(x[:5000]).argmax(1)
        acc = (pred == y[:5000]).float().mean().item()
    history_torch.append({'epoch': epoch + 1, 'loss': sum(losses) / len(losses), 'acc_sample': acc})
    print(history_torch[-1])

100%|██████████| 206M/206M [00:13<00:00, 14.8MB/s]


{'epoch': 1, 'loss': 2.136100895364176, 'acc_sample': 0.21580000221729279}
{'epoch': 2, 'loss': 2.0042445530945603, 'acc_sample': 0.1907999962568283}
{'epoch': 3, 'loss': 1.9561988772316412, 'acc_sample': 0.17020000517368317}
{'epoch': 4, 'loss': 1.8578879906033927, 'acc_sample': 0.3001999855041504}
{'epoch': 5, 'loss': 1.7712226974015886, 'acc_sample': 0.3222000002861023}
{'epoch': 6, 'loss': 1.736183517019857, 'acc_sample': 0.30000001192092896}
{'epoch': 7, 'loss': 1.7129727861082011, 'acc_sample': 0.2921999990940094}
{'epoch': 8, 'loss': 1.7051710347560318, 'acc_sample': 0.3240000009536743}
{'epoch': 9, 'loss': 1.684947325255383, 'acc_sample': 0.29600000381469727}
{'epoch': 10, 'loss': 1.6720992399548942, 'acc_sample': 0.33000001311302185}
{'epoch': 11, 'loss': 1.6742746769027277, 'acc_sample': 0.35440000891685486}
{'epoch': 12, 'loss': 1.6418095073578032, 'acc_sample': 0.36320000886917114}
{'epoch': 13, 'loss': 1.6436367338015274, 'acc_sample': 0.36880001425743103}
{'epoch': 14, 'l

Plote aqui as curvas NumPy e PyTorch sobrepostas e registre se a diferenca final de acuracia ficou em ate 2 pontos percentuais.